# INIT

In [0]:
from Dictionary import column_names
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


# Data Cleaning !!

## Update the column names

In [0]:
df= spark.table("vivo.bronze.cust_info")
for names in df.schema.fieldNames():
    df = df.withColumnRenamed(names, column_names[names])

## Removing unwanted spaces

In [0]:

for names in df.schema.fieldNames():
    df= df.withColumn(names, trim(names))

## Normalize the Values 

In [0]:
df= df.withColumn('marital_status', when(upper(col('marital_status')) == 'S', 'Single')\
              .when(upper(col('marital_status')) == 'M', 'Married').otherwise('N/A'))

In [0]:
df= df.withColumn('gender', when(upper(col('gender')) == 'M', 'Male')\
              .when(upper(col('gender')) == 'F', 'Female').otherwise('N/A'))

## Cleaning Duplicate/Null values in the Primary Key column(customer_id) 

In [0]:
win = Window.partitionBy("customer_id").orderBy("customer_onboarding_date")
df= df.withColumn("rank", row_number().over(win)).where((col("rank")== 1) & (col("customer_id").isNotNull())).select("customer_id", "customer_key", "firstname", "lastname", "marital_status", "gender", "customer_onboarding_date")



In [0]:
df.write.mode("overwrite").option("header", "true").saveAsTable("vivo.silver.cust_info")

# Customer Table Cleanup done !!

# Clean Product Table....

In [0]:
%sql
select * from vivo.bronze.prd_info

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,null,R,2003-07-01,null
211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,null,R,2003-07-01,null
212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12,S,2011-07-01,2007-12-28
213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14,S,2012-07-01,2008-12-27
214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13,S,2013-07-01,null
215,AC-HE-HL-U509,Sport-100 Helmet- Black,12,S,2011-07-01,2007-12-28
216,AC-HE-HL-U509,Sport-100 Helmet- Black,14,S,2012-07-01,2008-12-27
217,AC-HE-HL-U509,Sport-100 Helmet- Black,13,S,2013-07-01,null
218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3,M,2011-07-01,2007-12-28
219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3,M,2011-07-01,2007-12-28


In [0]:
dfp= spark.table("vivo.bronze.prd_info")

In [0]:
dfp.groupBy("prd_id").count().where((col("count") > 1) | (col('prd_id').isNull())).show()

+------+-----+
|prd_id|count|
+------+-----+
+------+-----+



## Updating the product key column

In [0]:
dfp= dfp.withColumn("prod_key", substring(col("prd_key"),7,length(col("prd_key"))))\
    .withColumn("prod_id", regexp_replace(substring(col("prd_key"),1,5),"-", "_"))\
        .select("prd_id", "prod_key","prod_id", "prd_nm", "prd_cost", "prd_line", "prd_start_dt", "prd_end_dt")



## Remove unwanted Space

In [0]:
for names in dfp.schema.fieldNames():
    dfp =dfp.withColumn(names, trim(col(names)))

## Correct the product end-date data !

In [0]:

wind= Window.partitionBy("prod_key").orderBy("prd_start_dt")
dfp= dfp.withColumn("prd_end_date",date_sub((lead("prd_start_dt").over(wind)),1))

## Normalizing the Product Line column

In [0]:
dfp= dfp.withColumn("prd_line", when(col("prd_line")=="M", "Mountain").when(col("prd_line")=="R", "Road").when(col("prd_line")=="S", "Standard").when(col("prd_line")=="T", "Touring").otherwise("N/A"))

## Handling NULL in Product Cost column

In [0]:
dfp=dfp.withColumn("prd_cost", when(col("prd_cost").isNull(),0).otherwise(col("prd_cost")))

In [0]:
dfp= dfp.select("prd_id", "prod_key", "prod_id", "prd_nm", "prd_cost", "prd_line", "prd_start_dt", "prd_end_date")

In [0]:
dfp.show()

+------+----------+-------+--------------------+--------+--------+------------+------------+
|prd_id|  prod_key|prod_id|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_date|
+------+----------+-------+--------------------+--------+--------+------------+------------+
|   601|   BB-7421|  CO_BB|   LL Bottom Bracket|      24|     N/A|  2013-07-01|        NULL|
|   602|   BB-8107|  CO_BB|   ML Bottom Bracket|      45|     N/A|  2013-07-01|        NULL|
|   603|   BB-9108|  CO_BB|   HL Bottom Bracket|      54|     N/A|  2013-07-01|        NULL|
|   478|   BC-M005|  AC_BC|Mountain Bottle Cage|       4|Mountain|  2013-07-01|        NULL|
|   479|   BC-R205|  AC_BC|    Road Bottle Cage|       3|    Road|  2013-07-01|        NULL|
|   596|BK-M18B-40|  BI_MB|Mountain-500 Blac...|     295|Mountain|  2013-07-01|        NULL|
|   597|BK-M18B-42|  BI_MB|Mountain-500 Blac...|     295|Mountain|  2013-07-01|        NULL|
|   598|BK-M18B-44|  BI_MB|Mountain-500 Blac...|     295|Mountain|  20

## Cleanup of the Product data is completed !!

## Create the Table

In [0]:
dfp.write.mode("overwrite").option("header", "true").saveAsTable("vivo.silver.prd_info")

# Sales Data Cleanup !!

In [0]:
dfs= spark.table("vivo.bronze.sales_details")

## Update all the Date Columns with coreect datatype and format

In [0]:
dfs=dfs.withColumn("sls_order_dt", when(length(col("sls_order_dt")) < 8, None).otherwise( to_date(col("sls_order_dt").cast("string"),"yyyyMMdd")))\
    .withColumn("sls_ship_dt", when(length(col("sls_ship_dt")) < 8, None).otherwise(to_date(col("sls_ship_dt").cast("string"),"yyyyMMdd")))\
        .withColumn("sls_due_dt", when(length(col("sls_due_dt")) < 8, None).otherwise(to_date(col("sls_due_dt").cast("string"),"yyyyMMdd")))

## Correct the total sales and Product price Values

In [0]:
dfs= dfs.withColumn("sls_sales", when((col("sls_sales").isNull() )| (col("sls_sales")<=0) | (col("sls_sales")!= col("sls_quantity")*abs(col("sls_price"))), col("sls_quantity") * abs(col("sls_price"))).otherwise(col("sls_sales")))\
    .withColumn("sls_price", when((col("sls_price").isNull()) | (col("sls_price")<=0), col("sls_sales")/col("sls_quantity")).otherwise( col("sls_price")))

## Data Cleanup completed for Sales Table!

## Create the Table

In [0]:
dfs.write.mode("overwrite").option("header", "true").option("inferSchema","true").saveAsTable("vivo.silver.sales_details")

# Clean the loc_a101

In [0]:
dfc = spark.table("vivo.bronze.loc_a101")

In [0]:
dfc.show()

+-----------+---------+
|        CID|    CNTRY|
+-----------+---------+
|AW-00011000|Australia|
|AW-00011001|Australia|
|AW-00011002|Australia|
|AW-00011003|Australia|
|AW-00011004|Australia|
|AW-00011005|Australia|
|AW-00011006|Australia|
|AW-00011007|Australia|
|AW-00011008|Australia|
|AW-00011009|Australia|
|AW-00011010|Australia|
|AW-00011011|Australia|
|AW-00011012|       US|
|AW-00011013|       US|
|AW-00011014|       US|
|AW-00011015|       US|
|AW-00011016|       US|
|AW-00011017|Australia|
|AW-00011018|Australia|
|AW-00011019|   Canada|
+-----------+---------+
only showing top 20 rows


## Creating the Data Frame

In [0]:
dfc= dfc.withColumn("CID", regexp_replace("CID","-", ""))


## Removing the Unwanted Space

In [0]:
for names in dfc.schema.fieldNames():
  dfc = dfc.withColumn(names, trim(names))

## Cleaning the blank data and normalizing the Coutnry values

In [0]:
dfc= dfc.withColumn("CNTRY", when(col("CNTRY").isin("USA", "US"), "United States")\
    .when(col("CNTRY")== "DE", "Germany")\
        .when(col("CNTRY")== "", None).otherwise(col("CNTRY")))

## Data cleaning completed !!

## Create the Table

In [0]:
dfc.write.mode("overwrite").option("header", "true").option("inferSchema", "true").saveAsTable("vivo.silver.cust_country")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4903256869506719>, line 1
----> 1 dfc.write.mode("overwrite").option("header", "true").option("inferSchema", "true").saveAsTable("vivo.silver.cust_country")

NameError: name 'dfc' is not defined

# Cleaning the CUST_az12 table

## Create the Dataframe

In [0]:
dfc2= spark.table("vivo.bronze.cust_az12")

In [0]:
dfc2.where(col("GEN").isNull()).show()

+-------------+----------+----+
|          CID|     BDATE| GEN|
+-------------+----------+----+
|NASAW00020354|1960-05-01|NULL|
|NASAW00020355|1959-10-29|NULL|
|NASAW00020356|1941-01-04|NULL|
|NASAW00020357|1941-01-28|NULL|
|NASAW00020358|1946-09-10|NULL|
|NASAW00020359|1940-08-31|NULL|
|NASAW00020360|1940-10-20|NULL|
|NASAW00020361|1942-04-02|NULL|
|NASAW00020362|1942-06-14|NULL|
|NASAW00020363|1941-09-02|NULL|
|NASAW00020364|1941-08-26|NULL|
|NASAW00020365|1966-01-24|NULL|
|NASAW00020366|1971-08-31|NULL|
|NASAW00020367|1976-08-16|NULL|
|NASAW00020368|1960-03-04|NULL|
|NASAW00020369|1965-08-19|NULL|
|NASAW00020370|1960-01-29|NULL|
|NASAW00020371|1959-08-17|NULL|
|NASAW00020372|1965-10-18|NULL|
|NASAW00020373|1965-01-13|NULL|
+-------------+----------+----+
only showing top 20 rows


## Clean the CID column

In [0]:
dfc2= dfc2.withColumn("CID", substring("CID", 4, length("CID")))

In [0]:
dfc2.where(col("BDATE") > current_date()).display()

CID,BDATE,GEN
NASAW00011257,2050-07-06,Female
NASAW00011410,2042-02-22,Male
NASAW00011551,2050-05-21,Male
NASAW00011562,2038-10-17,Male
NASAW00011581,2045-03-03,Female
NASAW00011775,2050-11-22,Female
NASAW00011912,2066-06-16,Female
NASAW00011915,9999-09-13,Male
NASAW00012123,2065-12-12,Male
NASAW00012188,9999-11-20,Female


## Handle the faulty data in the BDATE column

In [0]:
dfc2= dfc2.withColumn("BDATE", when(col("BDATE") > current_date(),None).otherwise(col("BDATE")))

## Remove unwanted Space from each columns

In [0]:
for name in dfc2.schema.fieldNames():
    dfc2= dfc2.withColumn(name,trim(name))

## Clean the Gender Column

In [0]:
dfc2= dfc2.withColumn("GEN", when(upper(col("GEN"))== "M", "Male").when(upper(col("GEN"))== "F", "Female").when((col("GEN")== "")| (col("GEN").isNull()),"N/A").otherwise(col("GEN") ))

In [0]:
dfc2.where(col("GEN")=="N/A").show()

+----------+----------+---+
|       CID|     BDATE|GEN|
+----------+----------+---+
|AW00011039|1954-06-16|N/A|
|AW00011040|1983-01-29|N/A|
|AW00011041|1983-04-15|N/A|
|AW00012145|1976-09-30|N/A|
|AW00020354|1960-05-01|N/A|
|AW00020355|1959-10-29|N/A|
|AW00020356|1941-01-04|N/A|
|AW00020357|1941-01-28|N/A|
|AW00020358|1946-09-10|N/A|
|AW00020359|1940-08-31|N/A|
|AW00020360|1940-10-20|N/A|
|AW00020361|1942-04-02|N/A|
|AW00020362|1942-06-14|N/A|
|AW00020363|1941-09-02|N/A|
|AW00020364|1941-08-26|N/A|
|AW00020365|1966-01-24|N/A|
|AW00020366|1971-08-31|N/A|
|AW00020367|1976-08-16|N/A|
|AW00020368|1960-03-04|N/A|
|AW00020369|1965-08-19|N/A|
+----------+----------+---+
only showing top 20 rows


## Data Cleaning is completed !

## Create Table

In [0]:
dfc2.write.mode("overwrite").option("inferSchema", "true").option("header", "True").saveAsTable("vivo.silver.cust_add_details")

# Create the Product Cat and Sub-Cat Table

In [0]:
dfpc= spark.table("vivo.bronze.px_cat_g1v2")

## Data is clean, No cleanup needed...

## Create the table !!

In [0]:
dfpc.write.mode("overwrite").option("inferSchema", "true").option("header", "True").saveAsTable("vivo.silver.product_Category")


# Silver Layer completed ! Table Clean up completed !!